In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense
import numpy as np
import matplotlib.pyplot as plt
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [2]:
corpus = '''
There are nights when the world forgets that it is spinning. The streets lie empty beneath the weight of indifferent stars, and the wind, ancient beyond memory, moves through abandoned places as though searching for names long erased from history. Every stone has witnessed an empire. Every tree has buried a generation beneath its roots. Yet neither speaks. Silence, it seems, is the oldest language.

A man walks alone through such a night believing himself unique in his loneliness, never suspecting that ten thousand others, scattered across mountains, cities, deserts, and oceans, carry the very same ache beneath different faces. He mistakes solitude for individuality. He mistakes suffering for ownership. But grief has no master. It passes from heart to heart like an unseen inheritance.

Civilizations have always imagined themselves immortal. They raise towers of glass, carve laws into marble, send machines beyond the clouds, and write their names across the sky with fire. Yet all things, no matter how magnificent, surrender eventually to dust. The libraries burn. The languages fracture. The monuments crack beneath seasons too patient to hate and too relentless to forgive. Time does not conquer. It simply waits.

There is an arrogance peculiar to youth that mistakes possibility for certainty. It believes every sunrise is a promise and every failure merely an inconvenience. Age knows better. Age understands that tomorrow is not owed but borrowed, that every familiar face is already halfway to becoming a memory, that the ordinary morning—the kettle boiling, birds arguing in the trees, sunlight slipping through cracked curtains—is a miracle so common it escapes applause.

History is filled not only with kings and revolutions but with invisible victories. A mother deciding not to surrender to despair. A stranger offering kindness without expectation. A teacher whose words quietly alter the course of a forgotten student's life. These moments never earn statues. No nation celebrates them. Yet they are the hidden architecture upon which the visible world rests.

People spend their lives searching for certainty as though truth were a destination instead of a direction. They collect beliefs the way travelers collect maps, convinced that possessing enough diagrams will spare them from becoming lost. But the universe offers no permanent roads. Even the stars drift. Even mountains migrate over ages too vast for memory. Stability is merely motion slowed beyond perception.

And still, despite everything, the human heart persists in hope. It plants gardens where wars have ended. It composes music after funerals. It laughs in hospitals. It falls in love beneath collapsing economies and darkening skies. There is something almost unreasonable about this refusal to surrender, something stubborn enough to appear divine. Perhaps hope is not optimism at all. Perhaps it is simply rebellion against inevitability.

When the final chapter closes—and it always closes without asking whether we are prepared—it is unlikely that anyone will remember the arguments won, the wealth accumulated, or the titles engraved upon polished doors. They will remember the warmth of a hand that refused to let go. The voice that stayed gentle when anger would have been easier. The presence that made the unbearable fractionally lighter.

In the end, perhaps greatness was never measured by how loudly one announced their existence, but by how quietly the world became kinder because they had passed through it. And if that is true, then history has overlooked its greatest heroes, for they were never seeking history at all. They were simply trying, one ordinary day after another, to become less afraid of being human.
'''
print(corpus)


There are nights when the world forgets that it is spinning. The streets lie empty beneath the weight of indifferent stars, and the wind, ancient beyond memory, moves through abandoned places as though searching for names long erased from history. Every stone has witnessed an empire. Every tree has buried a generation beneath its roots. Yet neither speaks. Silence, it seems, is the oldest language.

A man walks alone through such a night believing himself unique in his loneliness, never suspecting that ten thousand others, scattered across mountains, cities, deserts, and oceans, carry the very same ache beneath different faces. He mistakes solitude for individuality. He mistakes suffering for ownership. But grief has no master. It passes from heart to heart like an unseen inheritance.

Civilizations have always imagined themselves immortal. They raise towers of glass, carve laws into marble, send machines beyond the clouds, and write their names across the sky with fire. Yet all thing

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])

total_words = len(tokenizer.word_index) + 1
print("Vocabulary size:", total_words)

input_sequences = []
for line in corpus.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)

Vocabulary size: 361
X shape: (574, 70)
y shape: (574,)


In [4]:
rnn_model = Sequential([
    Embedding(total_words, 128, input_length=max_len-1),
    SimpleRNN(128),
    Dense(total_words, activation='softmax')
])

rnn_model.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

rnn_history = rnn_model.fit(X, y, epochs=200, verbose=0)
print("Vanilla RNN training completed")

C:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Vanilla RNN training completed


In [5]:
lstm_model = Sequential([
    Embedding(total_words, 128, input_length=max_len-1),
    LSTM(128),
    Dense(total_words, activation='softmax')
])

lstm_model.compile(loss='sparse_categorical_crossentropy',
                   optimizer='adam',
                   metrics=['accuracy'])

lstm_history = lstm_model.fit(X, y, epochs=200, verbose=0)
print("LSTM training completed")

LSTM training completed


In [ ]:
gru_model = Sequential([
    Embedding(total_words, 128, input_length=max_len-1),
    GRU(128),
    Dense(total_words, activation='softmax')
])

gru_model.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

gru_history = gru_model.fit(X, y, epochs=200, verbose=0)
print("GRU training completed")

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(rnn_history.history['loss'], label='RNN')
plt.plot(lstm_history.history['loss'], label='LSTM')
plt.plot(gru_history.history['loss'], label='GRU')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Comparison")
plt.legend()
plt.show()

In [ ]:
def generate_text(model, seed_text, next_words=5):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

In [ ]:
print("RNN :", generate_text(rnn_model, "deep learning", 10))
print("LSTM:", generate_text(lstm_model, "deep learning", 10))
print("GRU :", generate_text(gru_model, "deep learning", 10))